General information

In [ ]:
# Time/year/month etc. use time_period as a parameter
# everything else, use filters as a parameter

Import libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

Import class

In [2]:
import sys
sys.path.insert(0, '..')

from src.fetcher import SSBcsvFetcher

fetcher = SSBcsvFetcher()

Read reference tabøe

In [9]:
reference = pd.read_csv(r'C:\Users\matia\Desktop\ssb\eda\reference_dfs\counties_2026.csv', sep=';', encoding='ISO-8859-1')

Get information

In [ ]:
# Print nicely formatted info
fetcher.print_table_info('07459')

Inspect metadata

In [ ]:
# See what filters are available
fetcher.print_dimensions('07459')

Inspect table in detail

In [ ]:
# inspect specific table in detail
fetcher.inspect_table('07459')

Fetch the desired data

In [3]:
# Define the 15 county codes
county_codes = ['03', '11', '15', '18', '31', '32', '33', '34', '39', '40', '42', '46', '50', '55', '56']

# Fetch raw data (not cleaned) to preserve full names
df = fetcher.fetch_clean_table(
    '07459',
    time_period='2025',
    filters={
        'ContentsCode': 'Personer1',
        'Region': ','.join(county_codes)
    }
)

INFO:src.api:Fetching dimensions for table 07459
INFO:src.api:Fetching pivot table 07459
INFO:src.api:URL: https://data.ssb.no/api/pxwebapi/v2/tables/07459/data?lang=no&outputformat=csv&outputformatparams=separatorsemicolon,usecodesandtexts&stub=Tid&heading=statistikkvariabel&valuecodes[Tid]=2025&valuecodes[ContentsCode]=Personer1&valuecodes[Region]=03%2C11%2C15%2C18%2C31%2C32%2C33%2C34%2C39%2C40%2C42%2C46%2C50%2C55%2C56&valuecodes[Kjonn]=*&valuecodes[Alder]=*
INFO:src.api:Successfully fetched 3180 rows


Clean DataFrame (downstream)

In [ ]:
# Clean the columns by removing the first code part
for col in ['region', 'kjønn', 'alder']:
    df[col] = df[col].astype(str).str.split(' - ', n=1).str[1].str.strip()

Create age-bins

In [18]:
# Create age bins
def assign_age_bin(age_str):
    age = int(age_str.split()[0])
    if age <= 5:
        return '0-5 år'
    elif age <= 15:
        return '6-15 år'
    elif age <= 19:
        return '16-19 år'
    elif age <= 24:
        return '20-24 år'
    elif age <= 49:
        return '25-49 år'
    elif age <= 66:
        return '50-66 år'
    elif age <= 79:
        return '67-79 år'
    else:
        return '80+ år'

# Apply binning
df['age_bin'] = df['alder'].apply(assign_age_bin)

# Group and sum by region_name, kjønn, and age_bin
df_binned = df.groupby(['region', 'kjønn', 'age_bin'])['2025'].sum().reset_index()
df_binned.columns = ['region', 'kjønn', 'age_bin', 'population']


# Remove region codes from region column using a regular expression
df['region'] = df['region'].str.replace(r'^\d+\s+', '', regex=True).str.strip()

# Rename specific regions
rename_map = {
    'Oslo - Oslove': 'Oslo',
    'Trööndelage': 'Trøndelag',
    'Nordland - Nordlánnda': 'Nordland',
    'Troms - Romsa - Tromssa': 'Troms',
    'Finnmark - Finnmárku - Finmarkku': 'Finnmark'
}

df['region'] = df['region'].replace(rename_map)

df

,region,kjønn,alder,2025,age_bin
0,Oslo,Kvinner,0 år,4359,0-5 år
1,Oslo,Kvinner,1 år,3831,0-5 år
2,Oslo,Kvinner,2 år,3508,0-5 år
3,Oslo,Kvinner,3 år,3824,0-5 år
4,Oslo,Kvinner,4 år,3553,0-5 år
...,...,...,...,...,...
3175,Finnmark,Menn,101 år,0,80+ år
3176,Finnmark,Menn,102 år,0,80+ år
3177,Finnmark,Menn,103 år,0,80+ år
3178,Finnmark,Menn,104 år,0,80+ år


Aggregate by age and region

In [24]:
# Aggregate by age_bin, region_name, and kjønn
aggregated = df_binned.groupby(['region', 'kjønn', 'age_bin'])['population'].sum().reset_index()

# Define age bin order for proper sorting
age_bin_order = ['0-5 år', '6-15 år', '16-19 år', '20-24 år', '25-49 år', '50-66 år', '67-79 år', '80+ år']
aggregated['age_bin'] = pd.Categorical(aggregated['age_bin'], categories=age_bin_order, ordered=True)
aggregated = aggregated.sort_values(['region', 'kjønn', 'age_bin'])

# Create age_number column
age_number_map = {
    '0-5 år': 1,
    '6-15 år': 2,
    '16-19 år': 3,
    '20-24 år': 4,
    '25-49 år': 5,
    '50-66 år': 6,
    '67-79 år': 7,
    '80+ år': 8
}

aggregated['age_number'] = aggregated['age_bin'].map(age_number_map)

# export aggregated dataframe to csv
aggregated.to_csv('norwegian_population_age_gender_per_county_2025_binned.csv', index=False)

Example, see screenshot in powerbi_examples directory